Sales Outreach Agent

In [1]:
from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
import os
from openai import AsyncOpenAI
from agents import OpenAIChatCompletionsModel, set_tracing_disabled

set_tracing_disabled(True)
Deepseek_API = os.getenv("API_TOKEN")
deepseek_client = AsyncOpenAI(api_key=Deepseek_API, base_url="https://api.deepseek.com")
deepseek_model = OpenAIChatCompletionsModel(
    model="deepseek-flash",
    openai_client=deepseek_client
)


In [3]:
# Define agent personas with distinct tones

professional_prompt = (
    "You are a professional sales agent at TechLink, "
    "an online education company offering AI-driven, career-focused learning programs. "
    "Your goal is to write serious, persuasive cold outreach emails targeted at professionals and educational institutions."
)

engaging_prompt = (
    "You are a witty, engaging sales agent at TechLink, "
    "an online education company offering AI-powered, upskilling programs for tech and business roles. "
    "Craft humorous, attention-grabbing cold emails that spark curiosity and encourage responses."
)

concise_prompt = (
    "You are a fast-moving, efficient sales agent at TechLink, "
    "an online education platform offering smart, AI-powered learning paths. "
    "Write short, compelling cold emails that quickly highlight value and drive action."
)

# Create three distinct AI agents

professional_agent = Agent(
    name = "Professional_Sales_Agent",
    instructions = professional_prompt,
    model = deepseek_model
)

engaging_agent = Agent(
    name = "Engaging_Sales_Agent",
    instructions = engaging_prompt,
    model = deepseek_model
)

concise_agent = Agent(
    name = "Concise_Sales_Agent",
    instructions = concise_prompt,
    model = deepseek_model
)

In [4]:
# Stream and display email content from an agent
email_result = Runner.run_streamed(professional_agent, input="Write a cold sales email")
async for event in email_result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

**Subject:** AI skills for [Company]—worth a quick call?

Hi [First Name],

I noticed [personalization: your team is hiring for AI/data roles / your institution is expanding tech programs]. The hard part isn’t access to AI tools—it’s building people who can actually apply them to real work.

TechLink delivers AI-driven, career-focused learning programs for professionals and institutions. We combine applied projects, expert mentors, and industry-aligned curriculum to help [team/students] build job-ready AI, data, and automation skills—without pulling people out of work for months.

We’ve helped [similar organization] achieve [specific outcome: e.g., upskill 200 employees, improve placement rates, reduce training time]. I’d like to explore whether we can do the same for you.

Would you be open to a 15-minute call next week? If it’s not a fit, I’ll share a relevant resource and get out of your inbox.

Best,
[Your Name]
[Title] | TechLink
[Phone] | [Email]

P.S. If you’re not the right per

In [5]:
import asyncio
# The email topic or objective
email_topic = "Write a cold sales email?"
# Run all sales agents concurrently to generate email variations
agent_outputs = await asyncio.gather(
    Runner.run(professional_agent, email_topic),
    Runner.run(engaging_agent, email_topic),
    Runner.run(concise_agent, email_topic),
)

#Extract final outputs from each agent
cold_emails = [response.final_output for response in agent_outputs]

#Display all generated emails
for idx, email in enumerate(cold_emails, start=1):
    print(f"\n------Email Option{idx} ---\n{email}\n")




------Email Option1 ---
Here are two cold outreach emails for TechLink—one for professionals/teams and one for educational institutions. Keep the placeholders and personalize the first line.

---

**Version 1: For professionals / corporate teams**

**Subject:** Closing the AI skills gap at [Company]

Hi [First Name],

I’m reaching out because [personalized trigger—e.g., your team is hiring for AI/data roles, you mentioned expanding digital capabilities, etc.]. At TechLink, we deliver AI-driven, career-focused learning programs that help professionals build job-ready AI skills without stepping away from work.

Unlike generic courses, our programs adapt to each learner’s role, use real-world projects, and focus on what employers actually need—[e.g., prompt engineering, data analysis, automation, machine learning].

For [Company], that could mean:
• Faster upskilling across teams
• Higher completion and retention than passive courses
• Measurable skill gains tied to business outcomes

Wo

In [6]:
print("No. of outputs", len(agent_outputs))


No. of outputs 3


In [7]:
#We need to create the most evaluator agent to choose the most appealing email
selector_prompt = (
    "You are a prospective customer receiving cold outreach emails. "
    "Choose the email you're most likely to respond to, based solely on content. "
    "Don't explain your choice—just return the best email as-is."
)
email_selector_agent = Agent(
    name="Email Selector Agent",
    instructions=selector_prompt,
    model=deepseek_model
)

#Evaluate all email options and pick the most compelling one
with trace("Evaluate and select best cold email"):
    agent_outputs = await asyncio.gather(
        Runner.run(professional_agent, email_topic),
        Runner.run(engaging_agent, email_topic),
        Runner.run(concise_agent, email_topic),
    )
    #Extract email outputs
    email_variants = [res.final_output for res in agent_outputs]
    # Combine all into a single prompt for the selector agent
    evaluation_prompt = "Evaluate the following cold sales emails:\n\n" + "\n\n---\n\n".join(email_variants)
    #Run selector agent
    selected_response = await Runner.run(email_selector_agent, evaluation_prompt)
    print("\n Best selected email response: \n")
    print(selected_response.final_output)



 Best selected email response: 

Subject: Is your team AI-ready, or just AI-curious?  
Preview: We make AI upskilling painless. Mostly.

Hi {{FirstName}},

If AI were a coworker, would your team promote it, avoid eye contact, or accidentally CC it on a confidential email?

At TechLink, we turn “AI-curious” teams into AI-competent ones with AI-powered upskilling programs for tech and business roles. No jargon bingo. No death-by-slides. Just practical training that makes your people faster, sharper, and slightly less likely to blame the algorithm.

We helped {{SimilarCompany}} upskill {{Number}} employees in {{SkillArea}}—with results they could actually measure.

Worth a 15-minute chat to see if we can do the same for {{Company}}?

Reply “AI” and I’ll send a one-pager. Reply “human” and I’ll send the short version. Reply “potato” and I’ll respect the chaos.

Cheers,  
{{YourName}}  
{{Title}}, TechLink  
{{CalendarLink}}

P.S. Not interested? Reply “unsubscribe” and I’ll disappear fast

In [21]:
import sendgrid
from agents import function_tool, Agent, trace
import os
from sendgrid.helpers.mail import Email, To, Content, Mail
#Define tool functions to send an email
@function_tool
def send_sales_email(email_body : str):
    """Send the provided email body to the sales prospect."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    sender = Email("anshumani.pathak@gmail.com")
    recipient = To("anshumani.pathak@gmail.com")
    content = Content("text/plain", email_body)
    email = Mail(sender, recipient, "Cold Outreach from TechLink", content).get()
    response = sg.client.mail.send.post(request_body=email)
    return {"status": "sent"}

In [18]:
send_sales_email

FunctionTool(name='send_sales_email', description='Send the provided email body to the sales prospect.', params_json_schema={'properties': {'email_body': {'title': 'Email Body', 'type': 'string'}}, 'required': ['email_body'], 'title': 'send_sales_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x13f5b25d0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [16]:
#Convert agents into tool
#Wrap agents as a callable tool

tool_pro = professional_agent.as_tool(
    tool_name="professional_email_generator",
    tool_description="generate a serious, professional cold email"
)
tool_fun = engaging_agent.as_tool(
    tool_name="witty_email_generator",
    tool_description="Creates a humorous and engaging sales email."
)

tool_brief = concise_agent.as_tool(
    tool_name="brief_email_generator",
    tool_description="Produces a short, direct sales email."
)

# Define the full toolset available to the manager agent
toolset = [tool_pro, tool_fun, tool_brief, send_sales_email]

In [22]:
#Define the Sales Manager Agent with tool-based instructions

# Sales Manager only uses tools — no email writing on its own
manager_prompt = (
    "You are a sales manager at TechLink, responsible for outbound outreach. "
    "You are provided with several email-writing tools, each with a unique style. "
    "Your task is to try all three once, evaluate which email is the best for the user, "
    "and then use the 'send_sales_email' tool to send only the best email. "
    "Do NOT generate your own email — only use tools."
)

sales_manager_agent = Agent(
    name = "Sales Manager",
    instructions = manager_prompt,
    tools=toolset,
    model = deepseek_model
)

In [23]:
# Use the Sales Manager to run the end-to-end orchestration
message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales Manager Workflow"):
    final_result = await Runner.run(sales_manager_agent, message)

In [25]:
from agents import Agent

# Subject Line Generator
subject_agent = Agent(
    name="SubjectLineWriter",
    instructions="Generate a subject line for a cold email that improves open rates.",
    model=deepseek_model
)

# HTML Converter Agent
html_agent = Agent(
    name="HTMLFormatter",
    instructions="Convert plain text emails into well-structured, visually clear HTML format.",
    model=deepseek_model
)

subject_tool = subject_agent.as_tool(
    tool_name="generate_subject", 
    tool_description="Generate a cold email subject line."
    )

html_tool = html_agent.as_tool(
    tool_name="convert_to_html", 
    tool_description="Convert text email to HTML."
    )

# Function Tool to Send Email via SendGrid
@function_tool
def send_email(subject: str, html_body: str):
     """Send an HTML email with subject and body to the recipient list."""
     sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
     sender = Email("anshulc55@gmail.com")
     recipient = To("anshulc55@icloud.com")
     content = Content("text/html", html_body)
     email = Mail(sender, recipient, subject, content).get()
     response = sg.client.mail.send.post(request_body=email)
     return {"status": "sent"}

# Combine all tools for formatting and sending
email_formatter_tools = [subject_tool, html_tool, send_email]
email_formatter_tools

[FunctionTool(name='generate_subject', description='Generate a cold email subject line.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x13f6f4950>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None),
 FunctionTool(name='convert_to_html', description='Convert text email to HTML.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAs

In [27]:
email_manager_prompt = (
        "You are responsible for taking a raw email body and sending it out. "
        "First, use the subject generator tool. Then, convert the email to HTML format. "
        "Finally, use the send_email tool to deliver the message."
    )

# HandOff Agent
email_manager_agent = Agent(
    name="EmailManager",
    instructions=email_manager_prompt,
    tools=email_formatter_tools,
    model=deepseek_model,
    handoff_description="Format and send an email using subject generation and HTML conversion."
)


# These are your content generators
email_generator_tools = [tool_pro, tool_fun, tool_brief]

In [28]:
# The Sales Manager Agent coordinates everything
sales_manager_prompt = (
        "You are a sales manager tasked with sending the best possible cold email from your company TechLink. "
        "Use the available email-writing tools to generate several options. "
        "Choose the most persuasive email. Then, hand off to the EmailManager agent to format and send the email."
    )

sales_manager_agent = Agent(
    name="SalesManager",
    instructions=sales_manager_prompt,
    tools=email_generator_tools,
    handoffs=[email_manager_agent],
    model=deepseek_model
)

# Run the Full Workflow
prompt = "Send out a cold sales email addressed to Dear CEO from Anshul."
with trace("Automated Sales Email Flow"):
    result = await Runner.run(sales_manager_agent, prompt)